In [ ]:
import os
import sys
import time


import pandas as pd
from langchain_openai import ChatOpenAI
from ragas import EvaluationDataset, RunConfig,evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, AnswerAccuracy,LLMContextPrecisionWithReference,ResponseRelevancy,ContextRelevance


experiments_dir = os.path.abspath('..')
if experiments_dir not in sys.path:
    sys.path.append(experiments_dir)
    
from research.experiments.ingestion.ingestion_pipeline_2_0 import IngPipeline
from research.experiments.rag.rag_pipeline_2_0 import RagPipeline

h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\transformers\utils\hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


http://192.168.1.69:8000/generate


In [2]:
ing_configs = {
    'chunk_size': 400,
    'embed_model': "intfloat/e5-large-v2",
    'chunking_approach': "recursive",
    'kbs_path': "../test_data",
    "ingest_pip_version": "2.0",
    "embed_table": "embeddings_table_v2_0"
}

rag_configs = {
    'system_prompt_rag': "You are a helpful assistant that provides accurate and concise answers based on the provided context.",
    'embeddings_model_id': "intfloat/e5-large-v2",
    'cross_encoder_id': "cross-encoder/ms-marco-MiniLM-L-6-v2",
    'top_k': 5,
    'temperature': 0,
    'ce_threshold': 0,
    'search_type': "cosine",
    'src': "test_source",
    'embed_table': "embeddings_table_v2_0", # ingest synch param
    "ingest_pip_version": "2.0",            # ingest synch param
    'rag_pip_version': "2.0",               # NOT YET SUPPORTED
    "embed_table": "embeddings_table_v2_0", # ingest synch param
    'chunk_size': 200,                      # ingest synch param
}

global_configs = {
    'chunking_approach': "recursive",
    'system_prompt_rag': "You are a helpful assistant that provides accurate and concise answers based on the provided context.",
    'embeddings_model_id': "intfloat/e5-large-v2",
    'cross_encoder_id': "cross-encoder/ms-marco-MiniLM-L-6-v2",
    'top_k': 5,
    'temperature': 0,
    'ce_threshold': 0,
    'search_type': "cosine",
    'src': "test_source",
    'embed_table': "embeddings_table_v2_0", # ingest synch param
    "ingest_pip_version": "2.0",            # ingest synch param
    'rag_pip_version': "2.0",               # NOT YET SUPPORTED
    "embed_table": "embeddings_table_v2_0", # ingest synch param
    'chunk_size': 200,                      # ingest synch param
}

db_database = os.getenv("DB_DATABASE")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")

DB_CONFIG = {
    "dbname": db_database,
    "user": db_user,
    "password": db_password,
    "host": db_host
}




q_set_thermo = [
    "What is entropy and how is it defined in statistical mechanics?",
    "What does the zeroth law of thermodynamics state?",
    "How is temperature defined in terms of entropy and energy?",
    "What is the difference between heat and work?",
    "What does the second law of thermodynamics imply about entropy?",
    "Describe the Carnot cycle and its significance.",
    "What is the equipartition theorem and what does it state for ideal gases?",
    "What is the Maxwell-Boltzmann distribution?",
    "What is meant by thermal equilibrium?",
    "How is the internal energy of an ideal gas related to temperature?",
    "What is the value of Planck's constant?",
    "What is speed of light in free space?"
]

a_set_thermo = [
    "Entropy is a measure of the number of microscopic configurations (Ω) corresponding to a macroscopic state. It is defined by the Boltzmann formula: S = k ln Ω, where k is Boltzmann's constant.",
    "The zeroth law states that if two systems are each in thermal equilibrium with a third system, then they are in thermal equilibrium with each other.",
    "Temperature is defined by the relation: 1/T = (∂S/∂U)_V, where S is entropy and U is internal energy at constant volume.",
    "Heat is energy transferred due to temperature difference, while work is energy transferred via a force acting through a distance.",
    "The second law states that the total entropy of an isolated system can never decrease over time. It tends to increase, reaching a maximum at equilibrium.",
    "The Carnot cycle is an idealized reversible cycle consisting of two isothermal and two adiabatic processes. It sets the upper limit for the efficiency of any heat engine operating between two temperatures.",
    "The equipartition theorem states that each quadratic degree of freedom contributes (1/2)kT to the average energy. For an ideal monatomic gas, the internal energy is U = (3/2)NkT due to three translational degrees of freedom.",
    "The Maxwell-Boltzmann distribution describes the statistical distribution of speeds in a gas. It predicts the number of particles with a given speed at a specific temperature, assuming classical, non-interacting particles.",
    "Thermal equilibrium occurs when two systems in contact no longer exchange heat, i.e., they have the same temperature.",
    "For a monatomic ideal gas, the internal energy is given by: U = (3/2)NkT, where N is the number of particles, k is Boltzmann's constant, and T is the absolute temperature.",
    "Planck's constant (h) is equal to 6.626 × 10⁻³⁴ J⋅s.",
    "The speed of light in free space is 2.9979×10⁸ m/s. This is given in the text as c."
]

q_set_contract = [
    "What is the employee's job title under this contract?",
    "When did the employment contract come into effect?",
    "Is the contract for a fixed or indefinite period?",
    "What is the length of the trial period?",
    "What is the gross monthly salary of the employee?",
    "What is the value of the daily food allowance?",
    "Are vacation and Christmas subsidies included in the base salary?",
    "Where is the employee expected to work from while teleworking?",
    "Who provides and maintains the telework equipment?",
    "What is the retention period for personal data after termination of the contract?"
]

a_set_contract = [
    "Algorithm Engineer 2.",
    "October 7th, 2024.",
    "It is for an indefinite period.",
    "90 days.",
    "€3,571.42.",
    "€7.63 per day.",
    "No, they are paid in addition under legal terms.",
    "Rua da Ponte, 166, Penafiel 4560-300 Portugal.",
    "The company provides and maintains the work instruments required for telecommuting.",
    "Tax-relevant data is kept for 12 years; other personal data for 18 months."
]
llm = "gemma3n:e2b"

q_set = q_set_thermo + q_set_contract
a_set = a_set_thermo + a_set_contract

In [3]:
ing_pipeline = IngPipeline(DB_CONFIG, ing_configs)
rag_pipeline = RagPipeline(DB_CONFIG, rag_configs)

DEBUG: Initialized IngPipeline with kbs_path='../test_data', embed_model_id='intfloat/e5-large-v2', chunk_size=400, chunking_approach='recursive'


In [4]:
ing_pipeline.ingest_pipeline()

DEBUG: Executed query to fetch unique documents
DEBUG: Found 4 unique documents
[('Concepts in Thermal Physics - S. Blundell, K. Blundell (Oxford, 2006) WW.pdf', 'intfloat/e5-large-v2', 200, 200, 'recursive', '2.0'), ('Concepts in Thermal Physics - S. Blundell, K. Blundell (Oxford, 2006) WW.pdf', 'intfloat/e5-large-v2', 400, 400, 'recursive', '2.0'), ('Mykola_Shumskiy_Employment_Agreement-1.pdf', 'intfloat/e5-large-v2', 200, 200, 'recursive', '2.0'), ('Mykola_Shumskiy_Employment_Agreement-1.pdf', 'intfloat/e5-large-v2', 400, 400, 'recursive', '2.0')]
DEBUG: Knowledge base path exists: ../test_data
Starting ingestion pipeline...
DEBUG: Processing knowledge base: test_kb at path ../test_data\test_kb

Processing knowledge base: test_kb
Checking document: Concepts in Thermal Physics - S. Blundell, K. Blundell (Oxford, 2006) WW.pdf
  -> Concepts in Thermal Physics - S. Blundell, K. Blundell (Oxford, 2006) WW.pdf already ingested with the same configuration.
Checking document: Mykola_Shumski

In [5]:
def run_rag(user_prompt,llm):
    t0 = time.time()
    rag_output = rag_pipeline.generate_rag(
        user_prompt = user_prompt,
        selected_kb = 'test_kb',
        llm = "gemma3n:e2b")
    t1 = time.time()
    t_rag = t1 - t0
    return rag_output,t_rag,llm

def process_rag_output(rag_output,user_prompt,llm,t_rag):

    chunks = []
    for i, chunk in enumerate(rag_output['selected_chunks']):
        chunk = {
            "document": chunk['document'],
            "pages": chunk['pages'],
            "text": chunk['text'],
            "document": chunk['document'],
            "similarity_score": chunk['similarity'],
            "ce_score": chunk['ce_score'],
        }
        chunks.append(chunk)

    output = {
        "user_prompt": user_prompt,
        "response_text": rag_output['response']['message']['content'],
        "document_&_pages":rag_output['document_pages'],
        "chunks": chunks,
        "t_semantic_search": rag_output['t_semantic_search'],
        "t_process_context": rag_output['t_process_context'],
        "rag_time": t_rag,
        "llm": llm,
        }
    return output

def run_experiment(user_prompt, llm, expected_answer):
    rag_output, t_rag, llm = run_rag(user_prompt, llm)
    output = process_rag_output(rag_output, user_prompt, llm,t_rag)
    output['expected_answer'] = expected_answer
    return output

def run_set_experiment(q_set, a_set, llm):
    output = []
    for i, question in enumerate(q_set):
        user_prompt = question
        expected_answer = a_set[i]
        result = run_experiment(user_prompt, llm, expected_answer)
        output.append(result)
    return output







def get_scores(chunks):
    """
    Extracts similarity and ce scores from the last set of chunks in the DataFrame.
    """
    similarity_scores = [chunk['similarity_score'] for chunk in chunks]
    ce_scores = [chunk['ce_score'] for chunk in chunks]
    return similarity_scores, ce_scores

def get_text(chunks):
    """
    Extracts text from the last set of chunks in the DataFrame.
    """
    return [chunk['text'] for chunk in chunks]

def post_process_output_for_ragas(df):
    """
    Post-processes the output DataFrame for RAGAS.
    This function can be customized to format or filter the DataFrame as needed.
    """
    
    df.rename(columns={
        'user_prompt':'user_input',
        'response_text':'response',
        'chunks_text':'retrieved_contexts',
        'expected_answer':'reference'
        }, inplace=True)
    
    
    return df[['user_input', 'response', 'retrieved_contexts', 'reference']] # might need to change this to include more columns if needed

def experiment_post_processing(experiemnt_run):
    """
    Post-processes the experiment run output to create a DataFrame with relevant columns.
    """
    
    df = pd.DataFrame(experiemnt_run)
    df['chunks_text'] = df['chunks'].apply(get_text)
    df['similarity_scores'], df['ce_scores'] = zip(*df['chunks'].apply(get_scores))
    df_ragas = post_process_output_for_ragas(df)
    return df,df_ragas




def evaluate_ragas(df,df_ragas,eval_llm):
    """
    Evaluates the RAGAS output DataFrame.
    This function can be customized to perform specific evaluations or metrics.
    """
    # Example evaluation: count correct responses
    eval_llm = ChatOpenAI(model=eval_llm)
    dataset = df_ragas.to_dict(orient='records')
    evaluation_dataset = EvaluationDataset.from_list(dataset)
    run_config = RunConfig(timeout=200, max_retries=3,max_workers=2) 
    evaluator_llm = LangchainLLMWrapper(eval_llm)
    
    result = evaluate(dataset=evaluation_dataset,
                  metrics=[ContextRelevance(),LLMContextRecall(), Faithfulness(), AnswerAccuracy(),LLMContextPrecisionWithReference(),ResponseRelevancy()],
                  llm=evaluator_llm,
                  run_config=run_config)
    result_df = result.to_pandas()
    result_df['eval_llm'] = eval_llm.model_name
    
    return result_df

In [6]:
experiemnt_run = run_set_experiment(q_set, a_set, llm)
df,df_ragas = experiment_post_processing(experiemnt_run)

#eval_llm = 'gpt-3.5-turbo-0125'
eval_llm = "gpt-4.1-nano"
df_test = evaluate_ragas(df,df_ragas,eval_llm) # might need to do sampling of evals

intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
http://192.168.1.69:8000/generate
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
http://192.168.1.69:8000/generate
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
http://192.168.1.69:8000/generate
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
http://192.168.1.69:8000/generate
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
http://192.168.1.69

Evaluating: 100%|██████████| 132/132 [04:36<00:00,  2.09s/it]


In [11]:
df_test

,user_input,retrieved_contexts,response,reference,nv_context_relevance,context_recall,faithfulness,nv_accuracy,llm_context_precision_with_reference,answer_relevancy,eval_llm
0,What is entropy and how is it defined in stati...,"[validity of the prevailing, but subsequently ...",Entropy is a quantity that describes the disor...,Entropy is a measure of the number of microsco...,1.00,1.000000,0.666667,0.25,0.333333,0.987326,gpt-4.1-nano
1,What does the zeroth law of thermodynamics state?,[formulated. Early workersin thermodynamics to...,The zeroth law of thermodynamics states that t...,The zeroth law states that if two systems are ...,0.00,1.000000,1.000000,1.00,0.750000,0.999999,gpt-4.1-nano
2,How is temperature defined in terms of entropy...,[that Z=2c o s h/parenleftbiggβ∆ 2/parenrightb...,"Based on the provided text, temperature is def...",Temperature is defined by the relation: 1/T = ...,0.00,1.000000,1.000000,0.00,0.950000,0.949389,gpt-4.1-nano
3,What is the difference between heat and work?,[result is the transfer of heat from acolder t...,Based on the provided text:\n\n* **Heat** is...,Heat is energy transferred due to temperature ...,0.00,1.000000,0.600000,0.25,0.833333,0.987814,gpt-4.1-nano
4,What does the second law of thermodynamics imp...,"[process, we are watching how a system prepare...",The second law of thermodynamics implies that ...,The second law states that the total entropy o...,0.25,1.000000,0.800000,0.75,1.000000,0.978875,gpt-4.1-nano
5,Describe the Carnot cycle and its significance.,[called a Carnot cycle and which is illustrate...,The Carnot cycle is a theoretical thermodynami...,The Carnot cycle is an idealized reversible cy...,1.00,1.000000,1.000000,0.75,0.950000,0.949240,gpt-4.1-nano
6,What is the equipartition theorem and what doe...,[19.3). The equipartition theorem then implies...,The equipartition theorem states that the mean...,The equipartition theorem states that each qua...,0.00,1.000000,1.000000,0.50,0.916667,0.907987,gpt-4.1-nano
7,What is the Maxwell-Boltzmann distribution?,[shown that the corresponding expression for t...,The Maxwell-Boltzmann distribution is a probab...,The Maxwell-Boltzmann distribution describes t...,1.00,1.000000,1.000000,1.00,1.000000,0.958103,gpt-4.1-nano
8,What is meant by thermal equilibrium?,[Some deﬁnitions 11.1.1 A system in thermal eq...,"According to the provided text, a system is in...",Thermal equilibrium occurs when two systems in...,0.00,1.000000,1.000000,0.25,1.000000,0.956039,gpt-4.1-nano
9,How is the internal energy of an ideal gas rel...,"[monatomic gas, the internal energy Uis due to...","Based on the provided text, the internal energ...","For a monatomic ideal gas, the internal energy...",0.75,1.000000,1.000000,0.25,0.833333,0.890608,gpt-4.1-nano


In [12]:
df_test_metrics = df_test[['nv_context_relevance', 'context_recall', 'faithfulness', 'nv_accuracy', 'llm_context_precision_with_reference', 'answer_relevancy','eval_llm']]

In [13]:
merged_df = df.join(df_test_metrics, how='inner')


In [14]:
merged_df

,user_input,response,document_&_pages,chunks,t_semantic_search,t_process_context,rag_time,llm,reference,retrieved_contexts,similarity_scores,ce_scores,nv_context_relevance,context_recall,faithfulness,nv_accuracy,llm_context_precision_with_reference,answer_relevancy,eval_llm
0,What is entropy and how is it defined in stati...,Entropy is a quantity that describes the disor...,"{'Concepts in Thermal Physics - S. Blundell, K...",[{'document': 'Concepts in Thermal Physics - S...,0.712452,0.043920,32.744819,gemma3n:e2b,Entropy is a measure of the number of microsco...,"[validity of the prevailing, but subsequently ...","[0.8352406276407529, 0.8376961199326277, 0.839...","[0.20502932, 0.20117868, 0.19876614, 0.1983058...",1.00,1.000000,0.666667,0.25,0.333333,0.987326,gpt-4.1-nano
1,What does the zeroth law of thermodynamics state?,The zeroth law of thermodynamics states that t...,"{'Concepts in Thermal Physics - S. Blundell, K...",[{'document': 'Concepts in Thermal Physics - S...,0.111551,0.029967,9.590052,gemma3n:e2b,The zeroth law states that if two systems are ...,[formulated. Early workersin thermodynamics to...,"[0.8374277439986174, 0.8338166006561103, 0.829...","[0.20682052, 0.20094728, 0.20063902, 0.1976339...",0.00,1.000000,1.000000,1.00,0.750000,0.999999,gpt-4.1-nano
2,How is temperature defined in terms of entropy...,"Based on the provided text, temperature is def...","{'Concepts in Thermal Physics - S. Blundell, K...",[{'document': 'Concepts in Thermal Physics - S...,0.114347,0.053499,12.291312,gemma3n:e2b,Temperature is defined by the relation: 1/T = ...,[that Z=2c o s h/parenleftbiggβ∆ 2/parenrightb...,"[0.8265783267342662, 0.8273184748244002, 0.827...","[0.20441242, 0.20157109, 0.19941053, 0.1990938...",0.00,1.000000,1.000000,0.00,0.950000,0.949389,gpt-4.1-nano
3,What is the difference between heat and work?,Based on the provided text:\n\n* **Heat** is...,"{'Concepts in Thermal Physics - S. Blundell, K...",[{'document': 'Concepts in Thermal Physics - S...,0.108792,0.035969,11.215607,gemma3n:e2b,Heat is energy transferred due to temperature ...,[result is the transfer of heat from acolder t...,"[0.8176241946699897, 0.8338838925020524, 0.840...","[0.20248553, 0.20019116, 0.19995318, 0.1990453...",0.00,1.000000,0.600000,0.25,0.833333,0.987814,gpt-4.1-nano
4,What does the second law of thermodynamics imp...,The second law of thermodynamics implies that ...,"{'Concepts in Thermal Physics - S. Blundell, K...",[{'document': 'Concepts in Thermal Physics - S...,0.113717,0.063176,12.261437,gemma3n:e2b,The second law states that the total entropy o...,"[process, we are watching how a system prepare...","[0.8366311332692331, 0.8343561038272328, 0.835...","[0.20452924, 0.20432362, 0.20347482, 0.1943635...",0.25,1.000000,0.800000,0.75,1.000000,0.978875,gpt-4.1-nano
5,Describe the Carnot cycle and its significance.,The Carnot cycle is a theoretical thermodynami...,"{'Concepts in Thermal Physics - S. Blundell, K...",[{'document': 'Concepts in Thermal Physics - S...,0.107525,0.042569,13.399783,gemma3n:e2b,The Carnot cycle is an idealized reversible cy...,[called a Carnot cycle and which is illustrate...,"[0.8565648726319793, 0.8484598230274124, 0.841...","[0.20470233, 0.20231573, 0.19905831, 0.1976811...",1.00,1.000000,1.000000,0.75,0.950000,0.949240,gpt-4.1-nano
6,What is the equipartition theorem and what doe...,The equipartition theorem states that the mean...,"{'Concepts in Thermal Physics - S. Blundell, K...",[{'document': 'Concepts in Thermal Physics - S...,0.107547,0.041003,11.681270,gemma3n:e2b,The equipartition theorem states that each qua...,[19.3). The equipartition theorem then implies...,"[0.8402843917356848, 0.8424808633956834, 0.842...","[0.20215996, 0.20201969, 0.20019802, 0.200168,...",0.00,1.000000,1.000000,0.50,0.916667,0.907987,gpt-4.1-nano
7,What is the Maxwell-Boltzmann distribution?,The Maxwell-Boltzmann distribution is a probab...,"{'Concepts in Thermal Physics - S. Blundell, K...",[{'document': 'Concepts in Ther

In [15]:
merged_df.to_csv("ragas_evaluation_results.csv", index=False)

In [60]:
index = -2

def print_row(index,df):
    print('user_input')
    print(df.iloc[index]['user_input'])
    print('='*100)
    print('retrieved_contexts')
    for context in df.iloc[index]['retrieved_contexts']:
        print(f"- {context}")
    print('='*100)
    print('response')
    print(df.iloc[index]['response'])
    print('='*100)
    print('reference')
    print(df.iloc[index]['reference'])
    print('='*100)
    print('nv_context_relevance')
    print(df.iloc[index]['nv_context_relevance'])
    print('='*100)
    print('context_recall')
    print(df.iloc[index]['context_recall'])
    print('='*100)
    print('faithfulness')
    print(df.iloc[index]['faithfulness'])
    print('='*100)
    print('nv_accuracy')
    print(df.iloc[index]['nv_accuracy'])
    print('='*100)
    print('llm_context_precision_with_reference')
    print(df.iloc[index]['llm_context_precision_with_reference'])
    print('='*100)
    print('answer_relevancy')
    print(df.iloc[index]['answer_relevancy'])



In [76]:
merged_df.iloc[1]['chunks'][0]

{'document': 'Concepts in Thermal Physics - S. Blundell, K. Blundell (Oxford, 2006) WW.pdf',
 'pages': [50],
 'text': 'formulated. Early workersin thermodynamics took the content of the zeroth law as so obvious it hardly needed stating, and you might well agree with them! Never- theless, the zeroth law gives us some justiﬁcation for how to actuallymeasure temperature: we place the body whose temperature needs to be measured in thermal contact with a second body which displays some property which has a well-known dependence on temperature and wait for them to come into thermal equilibrium. The second body is called athermometer . The zeroth law then guarantees that if we have cal- ibrated this second body against any other standard thermometer, we should always get consistent results. Thus, a more succinct statement of the zeroth law 3is: ‘thermometers work’.3This version is from our colleague M.G. Bowler. 4.2 Thermometers We now make some remarks concerning thermometers. •For a thermom

In [61]:
index = 1
print_row(index, merged_df)

user_input
What does the zeroth law of thermodynamics state?
retrieved_contexts
- formulated. Early workersin thermodynamics took the content of the zeroth law as so obvious it hardly needed stating, and you might well agree with them! Never- theless, the zeroth law gives us some justiﬁcation for how to actuallymeasure temperature: we place the body whose temperature needs to be measured in thermal contact with a second body which displays some property which has a well-known dependence on temperature and wait for them to come into thermal equilibrium. The second body is called athermometer . The zeroth law then guarantees that if we have cal- ibrated this second body against any other standard thermometer, we should always get consistent results. Thus, a more succinct statement of the zeroth law 3is: ‘thermometers work’.3This version is from our colleague M.G. Bowler. 4.2 Thermometers We now make some remarks concerning thermometers. •For a thermometer to work well, its heat capacity 

In [65]:
index = 1
print_row(index, merged_df)

user_input
What does the zeroth law of thermodynamics state?
retrieved_contexts
- formulated. Early workersin thermodynamics took the content of the zeroth law as so obvious it hardly needed stating, and you might well agree with them! Never- theless, the zeroth law gives us some justiﬁcation for how to actuallymeasure temperature: we place the body whose temperature needs to be measured in thermal contact with a second body which displays some property which has a well-known dependence on temperature and wait for them to come into thermal equilibrium. The second body is called athermometer . The zeroth law then guarantees that if we have cal- ibrated this second body against any other standard thermometer, we should always get consistent results. Thus, a more succinct statement of the zeroth law 3is: ‘thermometers work’.3This version is from our colleague M.G. Bowler. 4.2 Thermometers We now make some remarks concerning thermometers. •For a thermometer to work well, its heat capacity 

In [70]:
index = 1
print_row(index, merged_df)

user_input
What does the zeroth law of thermodynamics state?
retrieved_contexts
- formulated. Early workersin thermodynamics took the content of the zeroth law as so obvious it hardly needed stating, and you might well agree with them! Never- theless, the zeroth law gives us some justiﬁcation for how to actuallymeasure temperature: we place the body whose temperature needs to be measured in thermal contact with a second body which displays some property which has a well-known dependence on temperature and wait for them to come into thermal equilibrium. The second body is called athermometer . The zeroth law then guarantees that if we have cal- ibrated this second body against any other standard thermometer, we should always get consistent results. Thus, a more succinct statement of the zeroth law 3is: ‘thermometers work’.3This version is from our colleague M.G. Bowler. 4.2 Thermometers We now make some remarks concerning thermometers. •For a thermometer to work well, its heat capacity 